# Store Sales – Time Series Forecasting (Favorita, Ecuador) · v11
**Business question:** how many units will each store sell in each product family over the next 16 days, so the chain can plan inventory and staffing?
**Metric:** RMSLE (models work on `log1p(sales)`, so RMSE on that scale = RMSLE).

**Version history**

| version | what changed | leaderboard RMSLE |
|---|---|---|
| v2 | lags ≥ 16 days, one validation window | 0.540 |
| v3 | features at the forecast origin + horizon, 3-fold CV | 0.430 |
| v4 | history from Aug 2015, XGBoost + HGB blend | 0.402 |
| v5 / v7 | several changes at once (data masking, categorical XGBoost, fold weighting) | 0.407 / 0.412 |
| v8-A | exact rebuild of v4 (pipeline is reproducible) | 0.402 |
| v9 | + Model B (per-family recursive LightGBM, 112 daily lags), 40/60 blend with v4 | 0.384 |
| v10 | Model B becomes an ensemble of 8 LightGBMs + cleaner inputs, 30/70 blend | 0.382 |
| v11 | Model B members re-chosen from the v10 hold-out results; longer look-backs added | – |

**Model A – direct multi-horizon (v4, unchanged):** one global model; features are summaries at the forecast origin plus the horizon 1–16.

**Model B – recursive, one model per product family:** 33 families × 54 stores. Each model forecasts one day ahead from recent daily sales, known-in-advance inputs and past transactions, then feeds its forecast back in as a lag, 16 times.

**What changed in v10 (ideas from public Kaggle notebooks that use the same model family – Ferdinand Berr's and Chong Zhen Jie's Darts + LightGBM notebooks – re-implemented here in NumPy + LightGBM):**
1. **Store closures are not demand.** Jan 1 and Dec 25, when almost every store is closed, are filled by interpolation instead of being kept as zeros, so they do not pollute the lags.
2. **Transactions:** a day with zero store sales gets 0 transactions; other gaps are interpolated.
3. **Ensemble inside Model B:** 4 look-back lengths (7, 56, 112, 365 days) × 2 training windows (all history, 2015 onward) = 8 LightGBM models per family, averaged on the log scale.
4. Smaller trees (max depth 6, learning rate 0.065) and a few more holiday flags (New Year, Mother's Day, Labour Day, All Souls' Day).

**What changed in v11 (based on the v10 hold-out, Jul 26 – Aug 10, 2017):**
- For every look-back length, training on **2015 onward** beat training on all history (e.g. 112 days: 0.3948 vs 0.3990), so the short look-backs now use only the 2015+ window.
- More variety in look-back length (7, 31, 56, 112, 365 and 730 days), like the 6-lag blend in Dmitry Shekhov's public notebook; the two longest also keep an all-history version.
- 7 models per family instead of 8, and the 2015+ models train on fewer rows, so the run is faster.

The A / B blend weight is picked on the same hold-out window.

Output files: `submission.csv` (blend with the best hold-out weight), `sub_B_only.csv` (Model B alone), `sub_blend_B085.csv` (15% A + 85% B).

⏱ About 1–1.5 h on Kaggle CPU – use **Save Version → Save & Run All**.

## Step 1 – Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

found = sorted(Path('/kaggle/input').rglob('train.csv'))   # finds the competition folder wherever Kaggle mounts it
DATA = found[0].parent
print('DATA =', DATA)

train    = pd.read_csv(DATA / 'train.csv', parse_dates=['date'])
test     = pd.read_csv(DATA / 'test.csv', parse_dates=['date'])
stores   = pd.read_csv(DATA / 'stores.csv')
oil      = pd.read_csv(DATA / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA / 'transactions.csv', parse_dates=['date'])

for name, d in [('train', train), ('test', test), ('stores', stores), ('oil', oil), ('holidays', holidays), ('transactions', transactions)]:
    print(f'{name:9s}', d.shape)
print('train dates:', train['date'].min().date(), '->', train['date'].max().date())
print('test  dates:', test['date'].min().date(), '->', test['date'].max().date())
print('stores:', train['store_nbr'].nunique(), '| families:', train['family'].nunique())
train.head()

## Step 2a – EDA: trend, weekday, promotion

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))

train.groupby('date')['sales'].sum().plot(ax=ax[0], lw=0.8)
ax[0].set_title('Total daily sales')

dow = train.groupby(train['date'].dt.dayofweek)['sales'].mean()
dow.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow.plot.bar(ax=ax[1]); ax[1].set_title('Average sales by weekday')

promo = train.groupby(train['onpromotion'] > 0)['sales'].mean()
promo.index = ['no promotion', 'on promotion']
promo.plot.bar(ax=ax[2]); ax[2].set_title('Average sales: promotion vs none')
plt.tight_layout(); plt.show()

print('Share of rows with zero sales:', round((train['sales'] == 0).mean(), 3))
print('Missing calendar days in train:',
      list(pd.date_range(train['date'].min(), train['date'].max()).difference(train['date'].unique()).date))

## Step 2b – EDA: stores, families, months

In [ ]:
tr = train.merge(stores, on='store_nbr', how='left')
fig, ax = plt.subplots(2, 2, figsize=(18, 10))

store_avg = tr.groupby('store_nbr')['sales'].mean().sort_values(ascending=False)
store_avg.plot.bar(ax=ax[0, 0], width=0.8)
ax[0, 0].set_title('Average sales per row, by store'); ax[0, 0].set_xlabel('store_nbr')

fam_avg = tr.groupby('family')['sales'].mean().sort_values()
fam_avg.plot.barh(ax=ax[0, 1], logx=True)
ax[0, 1].set_title('Average sales by family (log scale)')

month_avg = tr.groupby(tr['date'].dt.month)['sales'].mean()
month_avg.plot.bar(ax=ax[1, 0]); ax[1, 0].set_title('Average sales by month (all years)')

by_year_month = tr.groupby([tr['date'].dt.year.rename('year'), tr['date'].dt.month.rename('month')])['sales'].mean().unstack('year')
by_year_month.plot(ax=ax[1, 1], marker='o')
ax[1, 1].set_title('Monthly average sales, one line per year')
plt.tight_layout(); plt.show()

print('Store size gap (largest / smallest avg sales):', round(store_avg.iloc[0] / store_avg[store_avg > 0].iloc[-1], 1), 'x')
print('\nAverage sales by store type:')
print(tr.groupby('type')['sales'].mean().sort_values(ascending=False).round(1))

fam_profile = pd.DataFrame({
    'mean_sales': tr.groupby('family')['sales'].mean(),
    'zero_share': tr.groupby('family')['sales'].apply(lambda s: (s == 0).mean()),
}).sort_values('mean_sales', ascending=False)
print('\nFamily profile (top 5 and bottom 5 by volume):')
print(pd.concat([fam_profile.head(), fam_profile.tail()]).round(3))

first_sale = tr[tr['sales'] > 0].groupby('store_nbr')['date'].min()
late = first_sale[first_sale > train['date'].min() + pd.Timedelta(days=30)]
print('\nStores that only start selling later (opened during the period):', dict(late.dt.date))

## Step 3 – Reshape into one series per store × family, define validation folds
Rows = days, columns = series, values = `log1p(sales)`. The full calendar is rebuilt so missing days (Dec 25) become gaps. Then we define validation folds that mimic the test setup (3 recent windows + the same season last year).

In [ ]:
full = pd.concat([train.drop(columns='id'), test.drop(columns='id')], ignore_index=True)
all_days = pd.date_range(full['date'].min(), full['date'].max())

y_wide     = np.log1p(full.pivot(index='date', columns=['store_nbr', 'family'], values='sales')).reindex(all_days)
promo_wide = np.log1p(full.pivot(index='date', columns=['store_nbr', 'family'], values='onpromotion')).reindex(all_days)

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)   # nanmean of all-NaN slices

H = 16                                   # forecast horizon (days)
LAST_TRAIN = train['date'].max()
day_idx = {d: i for i, d in enumerate(all_days)}
LAST_IDX = day_idx[LAST_TRAIN]
Y = y_wide.to_numpy(dtype='float32')     # days x series, log1p(sales), NaN for test days / Dec 25
P = promo_wide.to_numpy(dtype='float32') # days x series, log1p(onpromotion)
cols = y_wide.columns

# Forecast origins are spaced 7 days apart so every origin falls on the same weekday as the real test origin.
# Each fold = forecast the 16 days after its origin, training only on origins whose targets end before it.
# 'last-year Aug' predicts Aug 17 - Sep 1, 2016: the same season as the test window (back to school).
FOLDS = {'fold 1': LAST_IDX - 21, 'fold 2': LAST_IDX - 35, 'fold 3': LAST_IDX - 49, 'last-year Aug': LAST_IDX - 364}
RECENT = ['fold 1', 'fold 2', 'fold 3']
for name, o in FOLDS.items():
    print(f'{name:14s}: origin {all_days[o].date()} -> predict {all_days[o+1].date()} .. {all_days[o+H].date()}')
print(f'{"test":14s}: origin {LAST_TRAIN.date()} -> predict {all_days[LAST_IDX+1].date()} .. {all_days[LAST_IDX+H].date()}')
print('series:', Y.shape[1])

## Step 4 – Hold-out window and Model A training set
Hold-out = forecast Jul 26 – Aug 10, 2017 from origin Jul 25 (same weekday as the real test origin, Aug 15).
Model A uses exactly the v4 features and training origins.

In [ ]:
VAL_ORIGIN = LAST_IDX - 21
print('hold-out: origin', all_days[VAL_ORIGIN].date(), '-> predict', all_days[VAL_ORIGIN+1].date(), '..', all_days[VAL_ORIGIN+H].date())
print('test    : origin', all_days[LAST_IDX].date(), '-> predict', all_days[LAST_IDX+1].date(), '..', all_days[LAST_IDX+H].date())

def first_k(h):                      # weeks back to the most recent observed same weekday
    return -(-h // 7)

st = stores.set_index('store_nbr')
ser_store = cols.get_level_values(0).to_numpy()
ser_family = cols.get_level_values(1).to_numpy()
ser_state = st.loc[ser_store, 'state'].to_numpy()
ser_city = st.loc[ser_store, 'city'].to_numpy()
fam_codes = pd.Categorical(ser_family).codes
store_codes = pd.Categorical(ser_store).codes
STATIC = {
    'store_code':      ser_store.astype(int),
    'family_code':     fam_codes,
    'store_type_code': pd.Categorical(st.loc[ser_store, 'type'].to_numpy()).codes,
    'city_code':       pd.Categorical(ser_city).codes,
    'cluster':         st.loc[ser_store, 'cluster'].to_numpy(),
}

hol = holidays[~holidays['transferred']]                     # transferred = moved away, so that date is normal
celebrated = hol['type'].isin(['Holiday', 'Transfer', 'Additional', 'Bridge'])
NAT_HOL  = set(hol.loc[(hol['locale'] == 'National') & celebrated, 'date'])
NAT_EVT  = set(hol.loc[(hol['locale'] == 'National') & (hol['type'] == 'Event'), 'date'])
WORK_DAY = set(hol.loc[hol['type'] == 'Work Day', 'date'])
REG_HOL  = hol[(hol['locale'] == 'Regional') & celebrated].groupby('date')['locale_name'].apply(set).to_dict()
LOC_HOL  = hol[(hol['locale'] == 'Local') & celebrated].groupby('date')['locale_name'].apply(set).to_dict()

def rows_(a, b):
    """Y rows a..b-1, clipped at the start of the data (avoids negative-index wrap-around)."""
    return Y[max(a, 0):max(b, 0)]

def group_mean(values, codes):
    """Average a per-series vector within each group (family or store) and broadcast back."""
    s = pd.Series(values).groupby(codes).transform('mean')
    return s.to_numpy(dtype='float32')

def make_block(o):
    """All rows for one forecast origin o (16 horizons x 1,782 series)."""
    win56 = Y[o-55:o+1]
    sold = np.nan_to_num(win56) > 0
    days_since = np.where(sold.any(0), sold[::-1].argmax(0), 56)
    mean_14 = np.nanmean(Y[o-13:o+1], 0)
    mean_56 = np.nanmean(win56, 0)
    ly = o - 364                                         # same weekday, 52 weeks earlier
    ly_before = np.nanmean(rows_(ly - 27, ly + 1), 0)
    ly_after = np.nanmean(rows_(ly + 1, ly + H + 1), 0)
    ly_lift = ly_after - ly_before                       # last year's change into the next 16 days
    trend_14 = mean_14 - mean_56
    at_origin = {
        'last_1':  Y[o],
        'mean_3':  np.nanmean(Y[o-2:o+1], 0),
        'mean_7':  np.nanmean(Y[o-6:o+1], 0),
        'mean_14': mean_14,
        'mean_28': np.nanmean(Y[o-27:o+1], 0),
        'mean_56': mean_56,
        'std_14':  np.nanstd(Y[o-13:o+1], 0),
        'zero_days_14':    (np.nan_to_num(Y[o-13:o+1], nan=1) == 0).sum(0),
        'days_since_sale': days_since,
        'promo_last_14':   np.nanmean(P[o-13:o+1], 0),
        'promo_next_16':   np.nansum(P[o+1:o+H+1], 0),
        'ly_lift':         ly_lift,
        'ly_lift_family':  group_mean(ly_lift, fam_codes),
        'trend_family':    group_mean(trend_14, fam_codes),
        'trend_store':     group_mean(trend_14, store_codes),
    }
    blocks = []
    for h in range(1, H + 1):
        t = o + h
        d = all_days[t]
        k0 = first_k(h)
        b = dict(at_origin, **STATIC)
        b['series'] = np.arange(Y.shape[1])
        b['origin'] = o
        b['date'] = d
        b['horizon'] = h
        b['dow_mean_4w'] = np.nanmean(Y[[t - 7 * k for k in range(k0, k0 + 4)]], 0)
        b['dow_mean_8w'] = np.nanmean(Y[[t - 7 * k for k in range(k0, k0 + 8)]], 0)
        b['dow_effect'] = b['dow_mean_8w'] - mean_56
        b['ly_same_day'] = Y[t - 364] if t - 364 >= 0 else np.full(Y.shape[1], np.nan, 'float32')
        b['ly_week_level'] = np.nanmean(rows_(t - 367, t - 360), 0) - ly_before
        b['promo'] = P[t]
        b['promo_vs_recent'] = P[t] - at_origin['promo_last_14']
        b['dow'] = d.dayofweek
        b['day'] = d.day
        b['is_payday'] = int(d.day == 15 or d.is_month_end)
        b['is_holiday'] = int(d in NAT_HOL)
        b['is_event'] = int(d in NAT_EVT)
        b['is_workday'] = int(d in WORK_DAY)
        b['is_regional_holiday'] = np.isin(ser_state, list(REG_HOL.get(d, ()))).astype(int)
        b['is_local_holiday'] = np.isin(ser_city, list(LOC_HOL.get(d, ()))).astype(int)
        b['y'] = Y[t]
        blocks.append(pd.DataFrame(b))
    return pd.concat(blocks, ignore_index=True)



start = day_idx[max(pd.Timestamp('2015-08-01'), all_days[0] + pd.Timedelta(days=60))]
quake_start, quake_end = day_idx[pd.Timestamp('2016-04-16')], day_idx[pd.Timestamp('2016-05-15')]
V4_ORIGINS = []
for k in range(3, (LAST_IDX - start) // 7 + 1):
    o = LAST_IDX - 7 * k
    if o + H >= quake_start and o + 1 <= quake_end:          # skip the April 2016 earthquake weeks
        continue
    if not (all_days[o] < pd.Timestamp('2017-01-01') and k % 2):
        V4_ORIGINS.append(o)                                  # v4: weekly in 2017, every 2 weeks before
V4_SET = set(V4_ORIGINS)

data = pd.concat([make_block(o) for o in sorted(V4_SET | {VAL_ORIGIN, LAST_IDX})], ignore_index=True)
num_cols = data.columns.difference(['date', 'series', 'origin', 'y'])
data[num_cols] = data[num_cols].astype('float32')

V4_FEATURES = ['last_1', 'mean_3', 'mean_7', 'mean_14', 'mean_28', 'mean_56', 'std_14', 'zero_days_14',
               'days_since_sale', 'dow_mean_4w', 'dow_mean_8w', 'dow_effect', 'horizon',
               'dow', 'day', 'is_payday', 'promo', 'promo_last_14', 'promo_next_16',
               'is_holiday', 'is_event', 'is_workday', 'is_regional_holiday', 'is_local_holiday'] + list(STATIC)
CAT_FEATURES = set(STATIC)
N_SERIES = Y.shape[1]
print(f'Model A: {len(V4_ORIGINS)} training origins | {len(data):,} rows | {len(V4_FEATURES)} features')

## Step 5 – Model A: direct multi-horizon model (v4 settings)
HGB 799 rounds + XGBoost 519 rounds, blended 25 / 75 on the log scale (settings found in v4's cross-validation).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
import xgboost as xgb

HGB_ROUNDS, XGB_ROUNDS, W_XGB = 799, 519, 0.75
XGB_PARAMS = dict(learning_rate=0.05, max_depth=9, min_child_weight=100, subsample=0.8, colsample_bytree=0.7,
                  reg_lambda=1.0, tree_method='hist', n_jobs=-1, random_state=42)

def model_a_forecast(origin):
    """Train on origins whose 16-day targets end before `origin`, forecast the 16 days after it.
    Returns log1p forecasts, shape (16 days, 1782 series)."""
    rows = data['origin'].isin(V4_SET) & data['origin'].le(origin - H) & data['y'].notna()
    X, y = data.loc[rows, V4_FEATURES], data.loc[rows, 'y']
    Xp = data.loc[data['origin'].eq(origin), V4_FEATURES]
    cat_idx = [i for i, f in enumerate(V4_FEATURES) if f in CAT_FEATURES]
    hgb = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=HGB_ROUNDS, max_leaf_nodes=127, min_samples_leaf=300,
                                        l2_regularization=1.0, categorical_features=cat_idx,
                                        early_stopping=False, random_state=42).fit(X, y)
    xgbm = xgb.XGBRegressor(n_estimators=XGB_ROUNDS, **XGB_PARAMS).fit(X, y)
    p = (1 - W_XGB) * np.clip(hgb.predict(Xp), 0, None) + W_XGB * np.clip(xgbm.predict(Xp), 0, None)
    return p.reshape(H, N_SERIES)

A_val = model_a_forecast(VAL_ORIGIN)
A_test = model_a_forecast(LAST_IDX)
print('Model A done')

## Step 6 – Model B inputs
- **Target:** log1p(sales); Dec 25 and zero-sales Jan 1 (stores closed) filled by interpolation; min-max scaled per series (scaler fitted only on data up to the forecast origin).
- **Known-in-advance inputs (window: 14 days before → target day):** promotions (scaled per series, plus 7- and 28-day averages), holidays for that store (national holiday, national event, earthquake, Christmas season, football, work day, New Year, Mother's Day, Labour Day, All Souls' Day, local/regional holiday), calendar (year, month, day, day of year, weekday, week of year, time trend), oil price and its 7/14/28-day averages.
- **Past-only input:** store transactions 16–23 days back (only past values are known, so the lags start at 16).
- **Store attributes:** one-hot store, city, state, type, cluster.

In [ ]:
N_DAYS = len(all_days)
store_ids = np.sort(stores['store_nbr'].unique())
store_pos = {s_: i for i, s_ in enumerate(store_ids)}
ser_store_pos = np.array([store_pos[s_] for s_ in ser_store])

# target: raw sales; store-closure days (Dec 25 is missing, Jan 1 with no sales and no promotion) -> interpolated
sales_raw = np.expm1(y_wide).copy()
promo_check = full.pivot(index='date', columns=['store_nbr', 'family'], values='onpromotion').reindex(all_days)[cols].fillna(0)
new_year = all_days[(all_days.month == 1) & (all_days.day == 1)]
closed_ny = (sales_raw.loc[new_year] == 0) & (promo_check.loc[new_year] == 0)
sales_raw.loc[new_year] = sales_raw.loc[new_year].mask(closed_ny)
print('Jan 1 store-family values treated as closures:', int(closed_ny.to_numpy().sum()), 'of', closed_ny.size)
sales_raw.iloc[:LAST_IDX + 1] = sales_raw.iloc[:LAST_IDX + 1].interpolate(limit_area='inside')
LOGS = np.log1p(sales_raw.to_numpy(dtype='float64'))                 # days x series

# promotions: scaled per series by its max, plus trailing 7/28-day means
promo_raw = full.pivot(index='date', columns=['store_nbr', 'family'], values='onpromotion').reindex(all_days)[cols].fillna(0)
pmax = promo_raw.max().replace(0, 1)
promo_s = promo_raw / pmax
PROMO = np.stack([promo_s.to_numpy(), promo_s.rolling(7, min_periods=1).mean().to_numpy(),
                  promo_s.rolling(28, min_periods=1).mean().to_numpy()], axis=-1).astype('float32')   # days x series x 3

# holidays per store
hol = holidays.copy()
desc = hol['description'].fillna('')
celebrated = hol['type'].isin(['Holiday', 'Transfer', 'Additional', 'Bridge']) & ~hol['transferred']
nat = hol['locale'] == 'National'
special = desc.str.contains('Terremoto|futbol', case=False)
day_flags = {
    'national_holiday': nat & celebrated,
    'national_event':   nat & (hol['type'] == 'Event') & ~special,
    'earthquake':       desc.str.contains('Terremoto', case=False),
    'christmas':        desc.str.contains('Navidad', case=False),
    'football':         desc.str.contains('futbol', case=False),
    'work_day':         hol['type'] == 'Work Day',
    'new_year':         nat & desc.str.contains('Primer dia del ano', case=False),
    'mothers_day':      nat & desc.str.contains('Madre', case=False),
    'labour_day':       nat & desc.str.contains('Trabajo', case=False),
    'all_souls_day':    nat & desc.str.contains('Difuntos', case=False),
}
G_HOL = np.stack([all_days.isin(hol.loc[m, 'date']) for m in day_flags.values()], axis=1).astype('float32')   # days x 10
st_info = stores.set_index('store_nbr').loc[store_ids]
LOCAL = np.zeros((N_DAYS, len(store_ids)), 'float32')
for j, (city, state) in enumerate(zip(st_info['city'], st_info['state'])):
    m = celebrated & (((hol['locale'] == 'Local') & (hol['locale_name'] == city)) |
                      ((hol['locale'] == 'Regional') & (hol['locale_name'] == state)))
    LOCAL[:, j] = all_days.isin(hol.loc[m, 'date'])
STORE_HOL = np.concatenate([np.repeat(G_HOL[:, None, :], len(store_ids), axis=1), LOCAL[:, :, None]], axis=-1)   # days x stores x 11

# calendar + oil (same for every series)
oil_d = oil.set_index('date')['dcoilwtico'].reindex(all_days).interpolate().bfill().ffill()
GLOBAL = np.column_stack([all_days.year, all_days.month, all_days.day, all_days.dayofyear, all_days.dayofweek,
                          all_days.isocalendar().week.to_numpy(), np.arange(N_DAYS), oil_d.to_numpy(),
                          oil_d.rolling(7, min_periods=1).mean(), oil_d.rolling(14, min_periods=1).mean(),
                          oil_d.rolling(28, min_periods=1).mean()]).astype('float32')    # days x 11

# transactions per store: 0 on days the store sold nothing, other gaps interpolated
store_sales_day = train.groupby(['date', 'store_nbr'])['sales'].sum().unstack().reindex(index=all_days, columns=store_ids)
tw = transactions.pivot(index='date', columns='store_nbr', values='transactions').reindex(index=all_days, columns=store_ids)
tw = tw.mask(store_sales_day.eq(0), 0)
tw.iloc[:LAST_IDX + 1] = tw.iloc[:LAST_IDX + 1].interpolate(limit_direction='both')
TRANS = tw.to_numpy(dtype='float32')                                    # days x stores

STATIC_OH = pd.get_dummies(st_info.reset_index()[['store_nbr', 'city', 'state', 'type', 'cluster']].astype(str)).to_numpy('float32')
FAMILIES = np.unique(ser_family)
FAM_IDX = {f: np.where(ser_family == f)[0] for f in FAMILIES}          # 54 series per family, in store order
FC_WIN, PC_LAGS = np.arange(-14, 1), np.arange(16, 24)
print('families:', len(FAMILIES), '| static one-hot columns:', STATIC_OH.shape[1])

## Step 7 – Model B: per-family recursive LightGBM ensemble
For every family: 7 models (look-back length × training window), each forecasting 16 days recursively; the 7 forecasts are averaged on the log scale.

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(n_estimators=100, learning_rate=0.065, max_depth=6, random_state=0, verbose=-1)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)          # silences sklearn's feature-name warning (it flooded the v10 log)

WINDOWS = {'all': None, '2015+': day_idx[pd.Timestamp('2015-01-01')]}
MEMBERS = [(7, '2015+'), (31, '2015+'), (56, '2015+'), (112, '2015+'), (365, '2015+'), (365, 'all'), (730, 'all')]

def family_inputs(f, cutoff):
    idx = FAM_IDX[f]
    seg = LOGS[:cutoff + 1, idx]
    mn, mx = np.nanmin(seg, 0), np.nanmax(seg, 0)
    rg = np.where(mx - mn > 0, mx - mn, 1.0)
    Yf = ((LOGS[:, idx] - mn) / rg).astype('float32')
    Yf[cutoff + 1:] = np.nan                                     # nothing after the origin is known
    pos = ser_store_pos[idx]
    FC = np.concatenate([PROMO[:, idx], STORE_HOL[:, pos], np.repeat(GLOBAL[:, None, :], len(idx), axis=1)], axis=-1)
    tmax = np.nanmax(TRANS[:cutoff + 1, pos], 0)
    TR = TRANS[:, pos] / np.where(tmax > 0, tmax, 1)
    return Yf, FC, TR, mn, rg

def make_rows(t_arr, Yf, FC, TR, lags):
    n, k = len(t_arr), Yf.shape[1]
    lag = Yf[t_arr[:, None] - np.arange(1, lags + 1)[None, :]].transpose(0, 2, 1).reshape(n * k, -1)
    fc = FC[t_arr[:, None] + FC_WIN[None, :]].transpose(0, 2, 1, 3).reshape(n * k, -1)
    pc = TR[t_arr[:, None] - PC_LAGS[None, :]].transpose(0, 2, 1).reshape(n * k, -1)
    X = np.concatenate([lag, fc, pc, np.tile(STATIC_OH, (n, 1))], axis=1)
    return X, Yf[t_arr].reshape(n * k)

def model_b_forecast(cutoff):
    """log1p forecasts for the 16 days after `cutoff` from every ensemble member: {member: (16, 1782)}."""
    out = {m: np.zeros((H, N_SERIES)) for m in MEMBERS}
    for i, f in enumerate(FAMILIES):
        Y0, FC, TR, mn, rg = family_inputs(f, cutoff)
        for lags, w in MEMBERS:
            t0 = max(lags, PC_LAGS.max(), WINDOWS[w] or 0)
            X, y = make_rows(np.arange(t0, cutoff + 1), Y0, FC, TR, lags)
            ok = ~np.isnan(y)
            model = lgb.LGBMRegressor(**LGB_PARAMS).fit(X[ok], y[ok])
            Yf = Y0.copy()
            for h in range(1, H + 1):                            # recursive: each forecast becomes a lag
                Xh, _ = make_rows(np.array([cutoff + h]), Yf, FC, TR, lags)
                Yf[cutoff + h] = model.predict(Xh)
            out[(lags, w)][:, FAM_IDX[f]] = np.clip(Yf[cutoff + 1:cutoff + H + 1] * rg + mn, 0, None)
        if i % 11 == 10:
            print(f'  {i + 1}/{len(FAMILIES)} families done')
    return out

B_val_members = model_b_forecast(VAL_ORIGIN)
B_val = np.mean(list(B_val_members.values()), axis=0)
print('Model B hold-out done')
B_test_members = model_b_forecast(LAST_IDX)
B_test = np.mean(list(B_test_members.values()), axis=0)
print('Model B test done')

## Step 8 – Hold-out comparison and blend weight
Zero rule (both models): a series with no sales in the 21 days before the origin is forecast as 0.

In [ ]:
def zero_rule(p, origin):
    dead = (np.nan_to_num(np.expm1(LOGS[origin - 20:origin + 1]), nan=0) == 0).all(0)
    return np.where(dead[None, :], 0, p)

truth = np.log1p(np.expm1(y_wide).to_numpy()[VAL_ORIGIN + 1:VAL_ORIGIN + H + 1])     # real sales, not interpolated
def rmsle(p):
    return float(np.sqrt(np.nanmean((zero_rule(p, VAL_ORIGIN) - truth) ** 2)))

base_last7 = np.tile(np.nanmean(Y[VAL_ORIGIN - 6:VAL_ORIGIN + 1], 0), (H, 1))
W_GRID = np.round(np.linspace(0, 1, 11), 1)
val_scores = pd.Series({w: rmsle((1 - w) * A_val + w * B_val) for w in W_GRID})
W_B = float(val_scores.idxmin())
print(f'Last 7-day mean baseline : {rmsle(np.nan_to_num(base_last7)):.4f}')
print(f'Model A (direct, v4)     : {val_scores[0.0]:.4f}')
print(f'Model B (7-model ensemble): {val_scores[1.0]:.4f}')
print('\nModel B members on the hold-out:')
print(pd.Series({f'lags {l:>3} | {w}': rmsle(p) for (l, w), p in B_val_members.items()}).round(4).to_string())
print()
print(f'Best blend (B weight {W_B}): {val_scores[W_B]:.4f}')

err = pd.DataFrame({'family': np.tile(ser_family, H),
                    'A': ((zero_rule(A_val, VAL_ORIGIN) - truth) ** 2).ravel(),
                    'B': ((zero_rule(B_val, VAL_ORIGIN) - truth) ** 2).ravel(),
                    'blend': ((zero_rule((1 - W_B) * A_val + W_B * B_val, VAL_ORIGIN) - truth) ** 2).ravel()})
by_family = err.groupby('family').mean().pow(0.5).sort_values('blend', ascending=False)
print('\nHold-out RMSLE by family (hardest 10):')
print(by_family.head(10).round(3))

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
val_scores.plot(ax=ax[0], marker='o'); ax[0].set_xlabel('weight on Model B'); ax[0].set_ylabel('hold-out RMSLE')
ax[0].set_title('Blend weight vs hold-out error')
by_family[['A', 'B']].plot.barh(ax=ax[1], figsize=(16, 8)); ax[1].set_title('Hold-out RMSLE by family')
plt.tight_layout(); plt.show()

## Step 9 – Write the submissions

In [ ]:
fc_days = all_days[LAST_IDX + 1:LAST_IDX + H + 1]
keys = pd.DataFrame({'date': np.repeat(fc_days.values, N_SERIES),
                     'store_nbr': np.tile(ser_store, H), 'family': np.tile(ser_family, H)})

def write(name, p_log):
    p_log = zero_rule(p_log, LAST_IDX)
    sub = test[['id', 'date', 'store_nbr', 'family']].merge(keys.assign(sales=np.expm1(p_log).ravel()),
                                                           on=['date', 'store_nbr', 'family'], how='left')[['id', 'sales']]
    assert len(sub) == len(test) and sub['sales'].notna().all()
    sub.to_csv(name, index=False)
    return sub

blend = (1 - W_B) * A_test + W_B * B_test
write('submission.csv', blend)
write('sub_B_only.csv', B_test)
write('sub_blend_B085.csv', 0.15 * A_test + 0.85 * B_test)
print('submission.csv uses Model B weight', W_B)
print('distance between A and B test forecasts (RMSLE):', round(float(np.sqrt(np.mean((A_test - B_test) ** 2))), 4))

actual = train[train['date'] > LAST_TRAIN - pd.Timedelta(days=45)].groupby('date')['sales'].sum()
ax = actual.plot(label='actual', figsize=(12, 4), color='black')
for name, p in [('Model A', A_test), ('Model B', B_test), ('blend', blend)]:
    pd.Series(np.expm1(zero_rule(p, LAST_IDX)).sum(1), index=fc_days).plot(ax=ax, label=name)
ax.legend(); ax.set_title('Total daily sales: recent actuals vs 16-day forecasts'); plt.show()

## Step 10 – Leaderboard results (fill in after submitting)

| file | what it is | hold-out RMSLE | leaderboard RMSLE |
|---|---|---|---|
| `submission.csv` | blend, weight picked on hold-out | | |
| `sub_B_only.csv` | Model B ensemble only | | |
| `sub_blend_B085.csv` | 15 / 85 blend | | |